# Turn AOPWiki queries into a SHACL catalogue

Keep a query’s title, description, prefixes and endpoint together. These examples are adapted from [AOP-Wiki-Queries](https://github.com/marvinm2/AOP-Wiki-Queries/tree/392f365865c387b62f3fea635f04be0de04ed519).

In [107]:
from rdfsolve import MinedSchema
from rdfsolve.api import QueryCollection
from rdfsolve.sparql_helper import SparqlHelper

schema = MinedSchema.from_json("../data/aopwikirdf.schema.json")
catalogue = QueryCollection()

## Add example queries

In [108]:
examples = [
    ("List all AOPs", "Return pathway identifiers and titles.", """
SELECT ?aop ?title WHERE {
    ?aop a <http://aopkb.org/aop_ontology#AdverseOutcomePathway> ;
         <http://purl.org/dc/elements/1.1/title> ?title .
}
"""),
    ("Export all chemicals", "Return chemical names and CAS registry numbers.", """
SELECT ?CAS ?ChemicalName WHERE {
    ?chemical a cheminf:000000 ;
        cheminf:000446 ?CAS ;
        dc:title ?ChemicalName .
}
"""),
]
for title, description, query in examples:
    catalogue.add(title, query, description=description,
                  endpoint=schema.about.endpoint, schema=schema)
list(catalogue.queries)

['List all AOPs', 'Export all chemicals']

## Inspect the typed SHACL

In [109]:
catalogue.shacl.queries[0]

ShaclSparqlExecutable(uri='urn:rdfsolve:query:f146113b90dab2371092b2a457f7513e1c5d0050e77de17b4c700963305efee2', text='\nSELECT ?aop ?title WHERE {\n    ?aop a <http://aopkb.org/aop_ontology#AdverseOutcomePathway> ;\n         <http://purl.org/dc/elements/1.1/title> ?title .\n}\n', query_type='SELECT', prefixes=['urn:rdfsolve:prefixes:481f7575d462e44ea183e42e06679b5497a4eddd2465bb5b47510597ad83c03f'], metadata={'http://www.w3.org/2000/01/rdf-schema#label': [RdfTerm(kind='literal', value='List all AOPs', datatype=None, language=None)], 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type': [RdfTerm(kind='uri', value='http://www.w3.org/ns/shacl#SPARQLExecutable', datatype=None, language=None), RdfTerm(kind='uri', value='http://www.w3.org/ns/shacl#SPARQLSelectExecutable', datatype=None, language=None)], 'http://www.w3.org/2000/01/rdf-schema#comment': [RdfTerm(kind='literal', value='Return pathway identifiers and titles.', datatype=None, language=None)], 'https://schema.org/target': [RdfTerm(ki

## Save the catalogue

Rdfsolve writes the executable queries and their `sh:prefixes` / `sh:declare` declarations.

In [110]:
turtle = catalogue.to_turtle("aopwiki-queries.ttl")
print(turtle)

@prefix aopwiki: <https://aopwiki.rdf.bigcat-bioinformatics.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<urn:rdfsolve:query:0e99ee1b2293a4933cb69587f45fdee1c118b3ac2f70cee78d87c177e6fd7705> a sh:SPARQLExecutable,
        sh:SPARQLSelectExecutable ;
    rdfs:label "Export all chemicals" ;
    rdfs:comment "Return chemical names and CAS registry numbers." ;
    sh:prefixes <urn:rdfsolve:prefixes:481f7575d462e44ea183e42e06679b5497a4eddd2465bb5b47510597ad83c03f> ;
    sh:select """
SELECT ?CAS ?ChemicalName WHERE {
    ?chemical a cheminf:000000 ;
        cheminf:000446 ?CAS ;
        dc:title ?ChemicalName .
}
""" ;
    schema:isBasedOn <urn:sha256:548844577d7ade2a5779040667ce3b4251447fb8b1eb9c5f9cbc0552f89b3b50> ;
    schema:target aopwiki:sparql .

<urn:rdfsolve:query:f146113b90dab2371092b2a457f7513e1c5d0050e77de17b4c700963305efee2> 

## Reopen a query

The prefixes are available when the query is read back.

In [111]:
saved = QueryCollection()
saved.load_shacl("aopwiki-queries.ttl")
print(saved.queries["Export all chemicals"].query)

PREFIX aopo: <http://aopkb.org/aop_ontology#>
PREFIX aopwiki: <https://aopwiki.rdf.bigcat-bioinformatics.org/>
PREFIX cheminf: <http://semanticscience.org/resource/CHEMINF_>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX edam_data: <http://edamontology.org/data_>
PREFIX edam_operation: <http://edamontology.org/operation_>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX go: <http://purl.obolibrary.org/obo/GO_>
PREFIX mmo: <http://purl.obolibrary.org/obo/MMO_>
PREFIX ncbitaxon: <http://purl.bioontology.org/ontology/NCBITAXON/>
PREFIX ncit: <http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX pato: <http://purl.obolibrary.org/obo/PATO_>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>



For a folder of `.rq` or `.sparql` files, use `catalogue.load_directory(folder, schema=schema, endpoint=schema.about.endpoint)`. Titles and descriptions are read from the query headers.

## Run the queries

With SparqlHelper:

In [112]:
helper = SparqlHelper(endpoint_url=schema.about.endpoint, timeout=30)
helper.queries = saved

In [113]:
helper.run_query("Export all chemicals")["results"]["bindings"][:10]

[{'CAS': {'type': 'literal', 'value': '341031-54-7'},
  'ChemicalName': {'type': 'literal', 'value': 'Sunitinib malate'}},
 {'CAS': {'type': 'literal', 'value': '100-00-5'},
  'ChemicalName': {'type': 'literal', 'value': '1-Chloro-4-nitrobenzene'}},
 {'CAS': {'type': 'literal', 'value': '100-42-5'},
  'ChemicalName': {'type': 'literal', 'value': 'Styrene'}},
 {'CAS': {'type': 'literal', 'value': '10028-15-6'},
  'ChemicalName': {'type': 'literal', 'value': 'Ozone'}},
 {'CAS': {'type': 'literal', 'value': '10102-43-9'},
  'ChemicalName': {'type': 'literal', 'value': 'Nitric oxide'}},
 {'CAS': {'type': 'literal', 'value': '10102-44-0'},
  'ChemicalName': {'type': 'literal', 'value': 'Nitrogen dioxide'}},
 {'CAS': {'type': 'literal', 'value': '10108-64-2'},
  'ChemicalName': {'type': 'literal', 'value': 'Cadmium chloride'}},
 {'CAS': {'type': 'literal', 'value': '10118-90-8'},
  'ChemicalName': {'type': 'literal', 'value': 'Minocycline'}},
 {'CAS': {'type': 'literal', 'value': '10161-33-8

Or with client:

In [114]:
from rdfsolve.client import Client

aopwiki = Client.open(schema)

In [115]:
helper.queries.queries["Export all chemicals"].query

'PREFIX aopo: <http://aopkb.org/aop_ontology#>\nPREFIX aopwiki: <https://aopwiki.rdf.bigcat-bioinformatics.org/>\nPREFIX cheminf: <http://semanticscience.org/resource/CHEMINF_>\nPREFIX dc: <http://purl.org/dc/elements/1.1/>\nPREFIX dcterms: <http://purl.org/dc/terms/>\nPREFIX edam_data: <http://edamontology.org/data_>\nPREFIX edam_operation: <http://edamontology.org/operation_>\nPREFIX foaf: <http://xmlns.com/foaf/0.1/>\nPREFIX go: <http://purl.obolibrary.org/obo/GO_>\nPREFIX mmo: <http://purl.obolibrary.org/obo/MMO_>\nPREFIX ncbitaxon: <http://purl.bioontology.org/ontology/NCBITAXON/>\nPREFIX ncit: <http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#>\nPREFIX owl: <http://www.w3.org/2002/07/owl#>\nPREFIX pato: <http://purl.obolibrary.org/obo/PATO_>\nPREFIX prov: <http://www.w3.org/ns/prov#>\nPREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX skos: <http://www.w3.org/2004/02/skos/core#>\nPREFIX xsd: <http://www.w3.org

In [116]:
aopwiki.select(
    aopwiki.prepare(
    helper.queries.queries["Export all chemicals"].query
));

In [117]:
aopwiki.query_log()

CAS,ChemicalName
341031-54-7,Sunitinib malate
100-00-5,1-Chloro-4-nitrobenzene
100-42-5,Styrene
10028-15-6,Ozone
10102-43-9,Nitric oxide
10102-44-0,Nitrogen dioxide
10108-64-2,Cadmium chloride
10118-90-8,Minocycline
10161-33-8,17beta-Trenbolone
102676-47-1,Fadrozole
